# 07 - Confronto dei risultati

Aggrega le tabelle prodotte dai notebook precedenti (`classical_baseline_results.csv`, `qsvc_ideal_results.csv`, `vqc_ideal_results.csv`) nelle tabelle e figure comparative del Capitolo 4 (sezione "Analisi comparativa trasversale"): confronto globale dell'accuratezza tra tutti i modelli e dataset, costo computazionale comparato e, se il notebook `06_hardware_execution.ipynb` è stato eseguito, degrado delle performance tra ambienti di esecuzione.

L'aggregazione vera e propria (tabella/figura `confronto_globale_accuratezza`, Figura `degrado_performance`) è delegata a `aggregate_ideal_results`/`update_degradation_figure` (`src/pipeline.py`), le stesse funzioni invocate da `run_pipeline.py --aggregate`: questo notebook si limita a caricare e combinare i risultati prodotti dai notebook 03-06 nel formato atteso da quelle funzioni, senza duplicarne la logica.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd

from src.evaluation.plots import plot_dataset_accuracy_comparison
from src.pipeline import aggregate_ideal_results, update_degradation_figure

TABLES_DIR = Path.cwd().parent / "results" / "tables"
FIGURES_DIR = Path.cwd().parent / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ORDER = ["Logistic Regression", "SVM (RBF kernel)", "Random Forest", "QSVC", "VQC"]

classical = pd.read_csv(TABLES_DIR / "classical_baseline_results.csv")
qsvc = pd.read_csv(TABLES_DIR / "qsvc_ideal_results.csv")
vqc = pd.read_csv(TABLES_DIR / "vqc_ideal_results.csv")

# accuracy_cv_std è presente solo per i modelli classici (si veda la
# sezione "Modelli classici di baseline" del Capitolo 4):
# usata come barra d'errore nella Figura wine_accuracy_comparison.
cost_cols = ["dataset", "model", "accuracy", "precision", "recall", "f1_score",
             "fit_time_s", "accuracy_cv_std"]
combined = pd.concat([
    classical.reindex(columns=cost_cols),
    qsvc.reindex(columns=cost_cols),
    vqc.reindex(columns=cost_cols),
], ignore_index=True)
combined

## Tabella `confronto_globale_accuratezza`

In [ ]:
global_accuracy = aggregate_ideal_results(combined, TABLES_DIR, FIGURES_DIR, model_order=MODEL_ORDER)
global_accuracy

In [ ]:
from IPython.display import Image

# La figura è già stata generata e salvata da aggregate_ideal_results
# (cella precedente); qui viene solo visualizzata.
Image(filename=str(FIGURES_DIR / "confronto_globale_accuratezza.png"))

## Figura `wine_accuracy_comparison`

In [ ]:
wine_rows = combined[combined["dataset"] == "wine"].fillna({"accuracy_cv_std": 0}).to_dict("records")
wine_plot_path = FIGURES_DIR / "wine_accuracy_comparison.png"
plot_dataset_accuracy_comparison(wine_rows, "Wine", wine_plot_path, model_order=MODEL_ORDER)
Image(filename=str(wine_plot_path))

## Tabella `costo_computazionale` (dataset Breast Cancer Wisconsin)

In [ ]:
cost_columns = ["model", "fit_time_s"]
bc_classical = classical[classical["dataset"] == "breast_cancer"][cost_columns]
bc_qsvc = qsvc[qsvc["dataset"] == "breast_cancer"][["model", "fit_time_s", "n_circuits", "circuit_depth"]]
bc_vqc = vqc[vqc["dataset"] == "breast_cancer"][["model", "fit_time_s", "n_circuits", "circuit_depth"]]

cost_table = pd.concat([bc_classical, bc_qsvc, bc_vqc], ignore_index=True)
cost_table.to_csv(TABLES_DIR / "costo_computazionale_breast_cancer.csv", index=False)
cost_table

## Figura `degrado_performance`

Confronta l'accuratezza di QSVC e VQC nei tre ambienti di esecuzione (ideale, simulazione con rumore, hardware reale), a partire dai risultati ideali aggregati sopra e, se disponibili, dalle tabelle `results_noisy_simulation.csv`/`results_real_hardware.csv` prodotte dal notebook `06_hardware_execution.ipynb`. Se quelle due tabelle non sono presenti (le celle corrispondenti del notebook 06 non sono ancora state eseguite, ad esempio per assenza di credenziali IBM Quantum Platform), la figura viene comunque generata con il solo ambiente ideale.

In [ ]:
from IPython.display import Image, display

degradation_plot_path = FIGURES_DIR / "degrado_performance_ambienti.png"
if update_degradation_figure(combined, TABLES_DIR, FIGURES_DIR):
    display(Image(filename=str(degradation_plot_path)))
else:
    print("Nessun risultato QSVC/VQC disponibile: eseguire prima le celle precedenti "
          "di questo notebook.")